## Download GHCNd station data

This script downloads or reads cached GHCN-Daily observations  
for nine primary weather stations in and around Minnesota.  
It organizes eleven target climatological variables on a common  
daily calendar from January 1, 1890 through the execution date.  
Actual data availability varies by station and variable.  

The script records quality-flag presence, preserves missing  
observations as NaN, and converts trace observations to zero.  
Quality filtering is applied during downstream analysis.  
Existing cached station files are reused unless refreshed.  

Known gaps in the MSP and International Falls records will  
be addressed through a separate donor-station splicing workflow.  

v1.1 designates a buildout of v1.0.   
    1.1 has sweeping data quality control checks and validations  
It is designed to support a reproducible workflow alongside  
the other v1.1 notebooks.  

In [1]:
import pandas as pd
import numpy as np
import xarray as xr
from tqdm import tqdm
from pathlib import Path

In [2]:
# create data directory structure
data_dir = Path("../station_data/ghcnd")
data_dir.mkdir(parents=True, exist_ok=True)

#controls refresh behavior, should usually be True
refresh_station_data=False

In [3]:
# station codes for the 9 first order stations
first_order_stations = ['USW00014926', 'USW00014922', #st cloud, mpls
                        'USW00014918', 'USW00014913', #intl falls, duluth
                        'USW00014925', 'USW00014916', #rochester, grand forks
                        'USW00014944', 'USW00014914', #sioux falls, fargo
                        'USW00014920']  #La crosse

donor_stations=[
    'USC00211448', #Chanhassen
    'USC00214809', #Littlefork
]

In [4]:
# relevant GHCNd variables
# convert to millimeters and Celsius
# precip and liquid precip equivalent are in tenths of millimeters
target_vars = ["TMAX", "TMIN", "PRCP", "MDPR", "DAPR",
               "SNOW", "MDSF", "DASF", "SNWD", "WESD", "WESF"]

temp_conv = 0.1
mm0p1 = 0.1
mm = 1
var_conv = {"TMAX": temp_conv,
            "TMIN": temp_conv,
            "PRCP": mm0p1,
            "MDPR": mm0p1,
            "DAPR": 1.0,
            "SNOW": mm,
            "SNWD": mm,
            "MDSF": mm,
            "DASF": 1.0,
            "WESD": mm0p1,
            "WESF": mm0p1}

# make a separate set of quality flag variables
qc_list = [f"{item}_q" for item in target_vars]

In [5]:
#matrix setup cell

# create a complete daily calende
target_dates = pd.date_range("1890-01-01", pd.Timestamp.today().normalize(), freq="D")

# data dimensions
n_days = len(target_dates)
n_stations = len(first_order_stations)

#all matrices use station rows and daily-date columns
data_matrices = {
    var: np.full((n_stations, n_days), np.nan, dtype=np.float32)
    for var in target_vars
}

qc_matrices = {
    var: np.zeros((n_stations, n_days), dtype=bool)
    for var in target_vars
}

#retain quality flag quality code identity along side bool qflag mask
qflag_matrices={
    var: np.full((n_stations, n_days), np.nan, dtype=object)
    for var in target_vars
}

mflag_matrices={
        var: np.full((n_stations, n_days), np.nan, dtype=object)
for var in target_vars
    }

sflag_matrices = {
    var: np.full((n_stations, n_days), np.nan, dtype=object)
    for var in target_vars
}

obs_time_matrices = {
    var: np.full((n_stations, n_days), np.nan, dtype=object)
    for var in target_vars
}

In [6]:
#cache raw donor records for the splicing notebook
for donor_station_id in donor_stations:
    donor_path=data_dir / f'{donor_station_id}.csv'

    if donor_path.exists() and not refresh_station_data:
        continue

    donor_url=(f'https://www.ncei.noaa.gov/data/global-historical-climatology-network-daily/access/{donor_station_id}.csv')

    try:
        donor_df=pd.read_csv(donor_url, low_memory=False)
        donor_df.to_csv(donor_path, index=False)
        print(f'Cached donor data: {donor_station_id}')

    except Exception as e:
        raise RuntimeError(
            f"Could not download or cache donor {donor_station_id}.") from e

Cached donor data: USC00211448
Cached donor data: USC00214809


In [ ]:
#preprocessing loop
#load and validate stations, then
#fill observation and attribute matrices
station_ids = []
lats = []
lons = []
station_names = []

#load stations, validate records, and fill the prepared arrays
for i, station_id in tqdm(enumerate(first_order_stations), total=len(first_order_stations)):
    # don't download station if it already exists
    # delete station file directory to re-download data (e.g. to get latest data)
    local_path = data_dir / f"{station_id}.csv"
    url = f"https://www.ncei.noaa.gov/data/global-historical-climatology-network-daily/access/{station_id}.csv"
    if local_path.exists() and not refresh_station_data:
        df = pd.read_csv(local_path, low_memory=False)
    else:
        try:
            df = pd.read_csv(url, low_memory=False)
            df.to_csv(local_path, index=False)
        except Exception as e:
            raise RuntimeError(
                f'could not download or cache {station_id}.'
                "station processing stopped before dataset construction"
            ) from e
#validate required columns and ensure observation rows exist
    #validate the station table before acessing its first row
    required_columns=['STATION', 'DATE', 'LATITUDE', 'LONGITUDE', 'NAME']
    missing_columns= [
        column for column in required_columns
        if column not in df.columns
    ]

    if missing_columns:
        raise ValueError(
            f'{station_id}: required columns are missing: {missing_columns}'
        )
    if df.empty:
        raise ValueError(
            f"{station_id}: the station table contains no observation rows"
        )
        
    # confirm that every record belongs to the requested station
    station_matches= df['STATION'].eq(station_id).fillna(False)

    if not station_matches.all():
        raise ValueError(
            f'{station_id}: the source contains a missing or unexpected station ID.'
        )
    #read and validate the metedata
    lat=pd.to_numeric(df['LATITUDE'].iloc[0], errors='coerce')
    lon=pd.to_numeric(df['LONGITUDE'].iloc[0], errors='coerce')
    name=df['NAME'].iloc[0]

    if pd.isna(lat) or not -90 <= lat <= 90:
        raise ValueError(f'{station_id}: missing or invalid latitude')

    if pd.isna(lon) or not -180 <= lon <= 180:
        raise ValueError(f'{station_id}: missing or invalid longitude')

    if pd.isna(name) or not str(name).strip():
        raise ValueError(f'{station_id}: missing station name')
    
    #parse dates abd report value that could not be interpreted
    parsed_dates=pd.to_datetime(df['DATE'], errors='coerce')
    invalid_dates= parsed_dates.isna()

    if invalid_dates.any():
        examples=df.loc[invalid_dates, 'DATE'].head(5).tolist()

        raise ValueError(
            f'{station_id}: {int(invalid_dates.sum())} invalid or missing dates. '
            f'Examples: {examples}'
        )

    df['DATE']=parsed_dates

    #remove exact duplicate rows, reporting those removals
    identical_rows=df.duplicated()

    if identical_rows.any():
        print(f'{station_id}: removing {int(identical_rows.sum())}'
        )
        df=df.loc[~identical_rows].copy()
    #start of repeated dates w/ conflicting records reporting
    conflicting_dates=df['DATE'].duplicated(keep=False)

    if conflicting_dates.any():
        examples=(
            df.loc[conflicting_dates, 'DATE']
            .astype(str)
            .drop_duplicates()
            .head(5)
            .tolist()
        )
        raise ValueError(
            f'{station_id}: conflicting records share the same date. '
            f'Examples:{examples}'
        )
    df=df.set_index('DATE')
    df_reindexed=df.reindex(target_dates)
#clean wach variable and retain its origional attribute fields
    # fill variable matrices
    for var in target_vars:
        if var in df_reindexed.columns:
            raw_vals= df_reindexed[var]         #start of unexpected text as var entries reporting
            vals= pd.to_numeric(raw_vals, errors='coerce')
            nonnumeric=raw_vals.notna() & vals.isna()

            if nonnumeric.any():
                examples=(
                    raw_vals.loc[nonnumeric]
                    .astype(str)
                    .drop_duplicates()
                    .head(5)
                    .tolist()
                )

                print(
                    f'{station_id}{var}: {int(nonnumeric.sum())}'
                    f'nonnumeric values converted to NaN. Examples:{examples}'
                )#end of uexpected text reporting


            # now grab and apply quality/measurement flags
            # grab attribute column
            attr_col = f"{var}_ATTRIBUTES"
            if attr_col in df_reindexed.columns:
                # split "mflag,qflag,sflag,obs_time" columns
                attrs = (
                    df_reindexed[attr_col]
                    .astype(str)
                    .str.split(",", expand=True)
                )
                # retain source flags and observation-time text
                for field_index, storage in (
                    (2, sflag_matrices[var]),
                    (3, obs_time_matrices[var])
                ):
                    field_values = (
                        attrs[field_index].astype("string").str.strip()
                        if field_index in attrs.columns
                        else pd.Series(
                            pd.NA,
                            index=df_reindexed.index,
                            dtype="string"
                        )
                    )

                    field_values = field_values.mask(
                        field_values.isin(["", "nan", "np.nan", "None"])
                    )

                    storage[i, :] = field_values.to_numpy(
                        dtype=object,
                        na_value=np.nan
                    )
                # measurement flag (mflag) is index 0 
                # quality flag (qflag) is index 1
                mflag = (
                    attrs[0].astype('string').str.strip()
                    if 0 in attrs.columns
                    else pd.Series(
                        pd.NA,
                        index=df_reindexed.index,
                        dtype='string'
                    )
                )
                mflag=mflag.mask(
                    mflag.isin(["", "nan", "np.nan", "None"])
                )
                #retain the trace indicator while storing numeric zero
                is_trace = mflag.eq('T').fillna(False)
                vals[is_trace] = 0

                # look for quality flags
                if 1 in attrs.columns:
                    qflag = attrs[1].astype('string').str.strip()
                    qflag=qflag.mask(
                        qflag.isin(["", "nan", "np.nan", "None"])
                    )
                else:
                    qflag = pd.Series(
                        pd.NA,
                        index=df_reindexed.index,
                        dtype='string'
                    )
            else:
                raise ValueError(
                    f'{station_id}: {var} exists, but {attr_col} is missing.'
                )
            # clean up fill values
            vals = vals.replace(-9999, np.nan)

            # apply unit conversion and save variable
            vals_converted = (var_conv[var] * vals).values.astype(np.float32)
            data_matrices[var][i, :] = vals_converted

            #store the boolean quality mask and origional flag codes
            # True (bad) False (passed)
            bad_q = qflag.notna()
            qc_matrices[var][i, :] = bad_q 
            
            qflag_matrices[var][i, :] = qflag.to_numpy(
                dtype=object,
                na_value=np.nan
            )
            mflag_matrices[var][i, :]=mflag.to_numpy(
                dtype=object,
                na_value=np.nan
            )           
    
    station_ids.append(station_id)
    lats.append(lat)
    lons.append(lon)
    station_names.append(name)



  0%|          | 0/9 [00:00<?, ?it/s]

In [ ]:
# save data and metadata
data_vars = {}

for var in target_vars:
    data_vars[var] = (("station", "time"), data_matrices[var])

    data_vars[f"{var}_q"] = (("station", "time"), qc_matrices[var])
    data_vars[f"{var}_qflag"]=(
        ('station', 'time'),
        qflag_matrices[var]
    )
    data_vars[f'{var}_mflag']= (
        ('station', 'time'),
        mflag_matrices[var]
    )
    data_vars[f"{var}_sflag"] = (
        ("station", "time"),
        sflag_matrices[var]
    )

    data_vars[f"{var}_obs_time"] = (
        ("station", "time"),
        obs_time_matrices[var]
    )
desc_txt = "tmax, tmin, precip (PRCP), snowfall (SNOW),"\
                "snow depth (SNWD), and snow water equivalent"\
                " of snowfall (WESF) and snowdepth (WESD)."\
                " Apply QC mask before using data, e.g."\
                " ds['SNOW_cleaned'] = ds['SNOW'].where(~ds['SNOW_q'])."
# package observations, quality masks, and origional attributes into xarray
ds = xr.Dataset(
    data_vars=data_vars,
    coords={
        "station": station_ids,
        "time": target_dates,
        "lat": ("station", lats),
        "lon": ("station", lons),
        "name": ("station", station_names),
    },
    attrs={
        "title": "GHCN-Daily Minnesota and bordering weather stations",
        "Description": desc_txt,
        "date_range": f"{target_dates[0].strftime('%Y-%m-%d')} to {target_dates[-1].strftime('%Y-%m-%d')}",
    },
)

In [ ]:

# Add the July-June reporting year used by the current analysis.
# July-December use next year; January-June keep their calendar year.
winter_year = ds.time.dt.year.where(
    ds.time.dt.month < 7,
    ds.time.dt.year + 1
)

ds = ds.assign_coords(winter_year=winter_year)

In [ ]:
# describe observation units and retained flag/time fields
ds['TMAX'].attrs = {"var": "daily max temperature", "units": "Celsius", "missing_value": np.nan}
ds['TMIN'].attrs = {"var": "daily min temperature", "units": "Celsius", "missing_value": np.nan}
ds['PRCP'].attrs = {"var": "daily total precipitation", "units": "mm", "missing_value": np.nan}
ds['MDPR'].attrs = {"var": "multi-day total precipitation", "units": "mm", "missing_value": np.nan}
ds['DAPR'].attrs = {"var": "multi-day total precipitation number of days summed over", "units": "days", "missing_value": np.nan}
ds['SNOW'].attrs = {"var": "daily total snowfall", "units": "mm", "missing_value": np.nan}
ds['MDSF'].attrs = {"var": "multi-day total snowfall", "units": "mm", "missing_value": np.nan}
ds['DASF'].attrs = {"var": "multi-day total snowfall number of days summed over", "units": "days", "missing_value": np.nan}
ds['SNWD'].attrs = {"var": "measured snow pack depth", "units": "mm", "missing_value": np.nan}
ds['WESD'].attrs = {"var": "snow water equivalent of snow pack", "units": "mm", "missing_value": np.nan}
ds['WESF'].attrs = {"var": "snow water equivalent of daily snowfall", "units": "mm", "missing_value": np.nan}

for var in qc_list:
    ds[var].attrs = {"var": f"{var} quality flag. A True value implies quality issues"}

for var in target_vars:
    ds[f'{var}_qflag'].attrs={ #metadata dictionary
        'var':f'{var} prigional GHCN-Daily quality code',
        'description': 'missing means no quality flag was recorded'
    }

    ds[f"{var}_mflag"].attrs = {
        'var': f'{var} origional GHCN-Daily measurment flag',
        'description': (
            'T identifies a trace; numeric trace observations are stpred as zero. '
            'Missing means no measurment flag was recorded'
        )
    }

    ds[f"{var}_sflag"].attrs = {
        "var": f"{var} original GHCN-Daily source flag",
        "description": "Source code retained as text."
    }

    ds[f"{var}_obs_time"].attrs = {
        "var": f"{var} original observation-time text",
        "description": (
            "Retained as supplied, including leading zeros. "
            "No timezone conversion or observation-date adjustment applied."
        )
    }

In [ ]:
#display nonmissing observation counts before quality filtering
station_df = ds.to_dataframe().reset_index()
display(station_df.groupby("station")[target_vars].count()) #table shows n of obs for each variable

In [ ]:
#report data coverage
coverage_rows=[]

for station_id, station_rows in station_df.groupby('station', sort=False):
    for var in target_vars:
        observed= station_rows[var].notna()
        flagged= observed & station_rows[f'{var}_q']
        usable= observed & ~station_rows[f'{var}_q']

        coverage_rows.append({
            "station": station_id,
            "name": station_rows["name"].iloc[0],
            "variable": var,
            "calendar_days": len(station_rows),
            "observed_days": int(observed.sum()),
            "missing_days": int((~observed).sum()),
            "flagged_observed_days": int(flagged.sum()),
            "usable_days": int(usable.sum()),
            "trace_days": int(station_rows[f"{var}_mflag"].eq("T").sum()),
            "first_observed": station_rows.loc[observed, "time"].min(),
            "last_observed": station_rows.loc[observed, "time"].max(),
            "last_usable": station_rows.loc[usable, "time"].max(),
        })
coverage_df = pd.DataFrame(coverage_rows)
display(coverage_df)
coverage_df.to_csv(data_dir / "station_variable_coverage.csv", index=False)

In [ ]:
#combine all station daily observations and retained flag/time columns into one csv
station_df.to_csv(data_dir/'all_station_data.csv', index=False)